# Notebook 2.6: Quantum State Tomography and Wigner Reconstruction
### Module 2: Phase Space and Quasiprobability Distributions (Module Capstone)
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_2_phase_space_quasiprobabilities/09_quantum_state_tomography.ipynb)

---

### Prerequisites
- Notebook 2.1: The Wigner Quasiprobability Distribution Fundamentals
- Notebook 2.2: Wigner Negativity as a Metric for Quantum Non-Classicality
- Notebook 2.5: Schrödinger Cat States and Phase Space Interference
- Notebook 0.3: Observables, Measurement, and Uncertainty Principles

### Pedagogical Learning Objectives
1. **The Experimental Tomography Problem:** Understand how optical homodyne tomography reconstructs the complete quantum density matrix $\hat{\rho}$ and 2D Wigner function $W(x, p)$ from 1D quadrature measurements.
2. **Balanced Homodyne Detection & Sinograms:** Model the detection apparatus where a local oscillator phase $\theta$ is swept across $[0, \pi)$, creating a projection sinogram $\mathcal{P}_\theta(x_\theta) = \langle x_\theta | \hat{\rho} | x_\theta \rangle$.
3. **The Projection-Slice Theorem & Inverse Radon Transform:** Derive the Filtered Backprojection (FBP) algorithm using the Ram-Lak ramp filter $|k|$ to invert the Radon transform.
4. **Maximum Likelihood Estimation (MaxLik / Hradil Algorithm):** Implement the iterative $R\rho R$ algorithm to guarantee physical, positive semidefinite reconstruction ($\hat{\rho} \ge 0, \mathrm{Tr}(\hat{\rho}) = 1$) from finite noisy data.
5. **Full Numerical Tomography Pipeline:** Synthesize realistic experimental homodyne datasets, reconstruct non-classical states (squeezed vacuum and Schrödinger cat), and evaluate fidelity $\mathcal{F} = \langle \psi_{\text{true}} | \hat{\rho}_{\text{recon}} | \psi_{\text{true}} \rangle > 0.98$.
6. **Module 2 Grand Capstone Synthesis:** Unify phase-space distributions ($W, Q, P$), non-classicality criteria, and experimental reconstruction methods.


In [ ]:
# Setup, environment detection, and library imports
import os
import sys
import math
import numpy as np
import scipy.linalg as la
from scipy.special import genlaguerre
import matplotlib.pyplot as plt

# Ensure matplotlib handles headless environments gracefully
if 'MPLBACKEND' not in os.environ:
    os.environ['MPLBACKEND'] = 'Agg' if not hasattr(sys, 'ps1') else 'module://matplotlib_inline.backend_inline'

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, IntSlider, FloatSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. Balanced Homodyne Detection and the Radon Transform

How can experimentalists determine the quantum state of a light beam without already knowing it?

In 1989, **K. Vogel and H. Risken** (*Phys. Rev. A* 40, 2847) proposed that an optical field could be completely reconstructed by measuring its rotated quadrature distributions. In 1993, **D. T. Smithey et al.** (*Phys. Rev. Lett.* 70, 1244) demonstrated this experimentally, giving birth to **Optical Homodyne Tomography (OHT)**!

---

### The Homodyne Measurement Mechanism

A signal field $\hat{a}$ interferes with an intense classical Local Oscillator (LO) $|\alpha_{\text{LO}} e^{i\theta}\rangle$ on a 50:50 beam splitter:
$$
\hat{x}_\theta = \frac{\hat{a} e^{-i\theta} + \hat{a}^\dagger e^{i\theta}}{\sqrt{2}} = \hat{x} \cos\theta + \hat{p} \sin\theta
$$
The difference photocurrent between the two balanced photodiodes is directly proportional to the rotated quadrature $\hat{x}_\theta$.

For a state $\hat{\rho}$, the probability of measuring value $x_\theta$ at LO phase $\theta$ is:

$$
\mathcal{P}_\theta(x_\theta) = \langle x_\theta | \hat{\rho} | x_\theta \rangle = \iint_{-\infty}^\infty W(x, p) \, \delta(x\cos\theta + p\sin\theta - x_\theta) \, dx \, dp
$$

This projection operation is mathematically identical to the **Radon Transform** $\mathcal{R}[W]$ used in medical X-ray computed tomography (CT scans)! The complete collection of marginals $\mathcal{P}(\theta, x_\theta)$ as a function of angle $\theta \in [0, \pi)$ is called a **Sinogram**.


## 2. Reconstruction Algorithms: Filtered Backprojection vs. MaxLik

Two principal algorithms exist to reconstruct the quantum state from measured homodyne marginals:

---

### 1. Filtered Backprojection (FBP / Inverse Radon Transform)

According to the **Fourier Projection-Slice Theorem**, the 1D Fourier transform of the marginal $\mathcal{P}_\theta(x_\theta)$ equals a radial slice of the 2D characteristic function:
$$
\tilde{\mathcal{P}}_\theta(k) = \int_{-\infty}^\infty \mathcal{P}_\theta(x_\theta) e^{-i k x_\theta} \, dx_\theta = \chi(k\cos\theta, k\sin\theta; 0)
$$
Inverting the 2D Fourier transform in polar coordinates yields the **Filtered Backprojection formula**:

$$
W(x, p) = \frac{1}{2\pi} \int_0^\pi d\theta \int_{-\infty}^\infty dk \, |k| \, \tilde{\mathcal{P}}_\theta(k) \, e^{i k (x\cos\theta + p\sin\theta)}
$$

where $|k|$ is the **Ram-Lak ramp filter**, which corrects for the over-sampling of low spatial frequencies near the origin.
- **Advantage:** Fast, direct, non-iterative linear transform.
- **Limitation:** In finite, noisy experimental datasets, FBP can produce minor unphysical ripples and small negative eigenvalues when converted to a density matrix.

---

### 2. Maximum Likelihood Estimation (MaxLik / Hradil Algorithm, 1997)

To guarantee that the reconstructed density matrix is strictly **physical** ($\hat{\rho} \ge 0$, $\hat{\rho}^\dagger = \hat{\rho}$, and $\mathrm{Tr}(\hat{\rho}) = 1$), **Zdeněk Hradil** (*Phys. Rev. A* 55, R1561, 1997) formulated the Maximum Likelihood problem:

Given $M$ measured quadrature samples $\{ (x_j, \theta_j) \}_{j=1}^M$, find the state $\hat{\rho}$ that maximizes the log-likelihood:
$$
\ln \mathcal{L}(\hat{\rho}) = \sum_{j=1}^M \ln \langle x_j, \theta_j | \hat{\rho} | x_j, \theta_j \rangle
$$
This is solved iteratively via the **$R\rho R$ algorithm**:

$$
\hat{R}(\hat{\rho}) = \frac{1}{M} \sum_{j=1}^M \frac{|x_j, \theta_j\rangle\langle x_j, \theta_j|}{\langle x_j, \theta_j | \hat{\rho} | x_j, \theta_j \rangle}
$$
$$
\hat{\rho}^{(k+1)} = \frac{\hat{R}(\hat{\rho}^{(k)}) \, \hat{\rho}^{(k)} \, \hat{R}(\hat{\rho}^{(k)})}{\mathrm{Tr}\left[ \hat{R}(\hat{\rho}^{(k)}) \, \hat{\rho}^{(k)} \, \hat{R}(\hat{\rho}^{(k)}) \right]}
$$
The sequence $\hat{\rho}^{(k)}$ converges monotonically to the physical maximum likelihood quantum state!


In [ ]:
# Synthetic Homodyne Detection Simulator and Sinogram Generation

def harmonic_quadrature_basis(N, x_pts):
    """Constructs QHO eigenstate spatial wavefunctions phi_n(x)."""
    H = np.zeros((N, len(x_pts)))
    H[0] = 1.0
    if N > 1:
        H[1] = 2.0 * x_pts
    for n in range(1, N - 1):
        H[n + 1] = 2.0 * x_pts * H[n] - 2.0 * n * H[n - 1]
        
    fact = np.array([float(math.factorial(n)) for n in range(N)])
    norm = 1.0 / np.sqrt((2.0**np.arange(N)) * fact * np.sqrt(np.pi))
    phi = np.zeros((N, len(x_pts)))
    for n in range(N):
        phi[n] = norm[n] * np.exp(-0.5 * x_pts**2) * H[n]
    return phi

def simulate_homodyne_sinogram(state, theta_angles, x_bins):
    """Computes exact marginal probability distributions P_theta(x_theta) for an array of LO angles."""
    if state.ndim == 1:
        psi = state / np.linalg.norm(state)
        rho = np.outer(psi, psi.conj())
    else:
        rho = state / np.real(np.trace(state))
        
    N = rho.shape[0]
    phi_x = harmonic_quadrature_basis(N, x_bins)
    
    sinogram = np.zeros((len(theta_angles), len(x_bins)))
    
    # Rotation operator in Fock basis: exp(-i * theta * n)
    n_diag = np.arange(N)
    for idx, th in enumerate(theta_angles):
        rot = np.exp(-1j * th * n_diag)
        # Rotated density matrix rho(theta) = U(th) rho U(th)^dag
        rho_th = (rot[:, None] * rho) * np.conj(rot[None, :])
        # Quadrature probability P_th(x) = sum_{n, m} rho_nm(th) phi_n(x) phi_m(x)
        # = diag(phi_x.T @ rho_th @ phi_x)
        p_dist = np.real(np.sum((phi_x.T @ rho_th) * phi_x.T, axis=1))
        p_dist = np.maximum(0.0, p_dist)
        sinogram[idx] = p_dist / np.sum(p_dist * (x_bins[1] - x_bins[0]))
        
    return sinogram

# Generate test target state: Squeezed Vacuum |0, r=0.65>
N_dim = 25
a_ann = np.diag(np.sqrt(np.arange(1, N_dim)), k=1)
r_sq = 0.65
S_op = la.expm(0.5 * r_sq * (a_ann@a_ann - a_ann.T.conj()@a_ann.T.conj()))
vac = np.zeros(N_dim, dtype=np.complex128); vac[0] = 1.0
psi_target = S_op @ vac

# Simulation parameters: 60 LO phase angles across [0, pi), 100 quadrature bins
n_angles = 60
angles = np.linspace(0, np.pi, n_angles, endpoint=False)
x_quad = np.linspace(-4.0, 4.0, 100)
dx_q = x_quad[1] - x_quad[0]

sinogram_sq = simulate_homodyne_sinogram(psi_target, angles, x_quad)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Sinogram P(theta, x_theta)
extent = [x_quad[0], x_quad[-1], 0, np.pi]
im = ax1.imshow(sinogram_sq, extent=extent, aspect='auto', cmap='plasma', origin='lower')
ax1.set_xlabel(r"Quadrature Amplitude $x_\theta$")
ax1.set_ylabel(r"Local Oscillator Phase $\theta$ (rad)")
ax1.set_title(r"Homodyne Sinogram $\mathcal{P}_\theta(x_\theta)$ (Squeezed Vacuum)")
plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04, label='Probability Density')

# Panel 2: Slices at theta = 0 (Squeezed quadrature) and theta = pi/2 (Anti-squeezed)
idx_0 = 0
idx_pi2 = len(angles) // 2
ax2.plot(x_quad, sinogram_sq[idx_0], 'b-', lw=2.5, label=r'$\theta = 0$ (Squeezed Quadrature $X_1$)')
ax2.plot(x_quad, sinogram_sq[idx_pi2], 'r--', lw=2.5, label=r'$\theta = \pi/2$ (Anti-squeezed Quadrature $X_2$)')

# Vacuum shot noise reference
p_shot = (1.0 / np.sqrt(np.pi)) * np.exp(-x_quad**2)
ax2.plot(x_quad, p_shot, 'k:', lw=1.8, label='Vacuum Shot Noise Reference')

ax2.set_xlim(-3.5, 3.5)
ax2.set_xlabel(r"Quadrature $x_\theta$")
ax2.set_ylabel(r"Probability $\mathcal{P}_\theta(x_\theta)$")
ax2.set_title("Quadrature Noise Profiles: Sub-Shot-Noise Reduction")
ax2.legend(loc='upper right', framealpha=0.9)
ax2.grid(True)

plt.tight_layout()
plt.show()

print("Sinogram generated: clear signature of squeezing at theta = 0 and anti-squeezing at theta = pi/2!")


In [ ]:
# Inverse Radon Transform via Filtered Backprojection (FBP)

def inverse_radon_fbp(sinogram, angles, x_bins, x_out, p_out):
    """Reconstructs 2D Wigner function W(x, p) from homodyne sinogram using Ram-Lak ramp filter."""
    N_angles, N_bins = sinogram.shape
    dx = x_bins[1] - x_bins[0]
    
    # 1. Fourier transform along quadrature bins: P_th(x) -> P_tilde_th(k)
    # Frequency axis k
    k_freq = 2.0 * np.pi * np.fft.fftfreq(N_bins, d=dx)
    
    # 2. Ram-Lak ramp filter with Hann window cutoff
    k_max = np.max(np.abs(k_freq)) * 0.75
    hann_window = 0.5 * (1.0 + np.cos(np.pi * np.clip(np.abs(k_freq) / k_max, 0, 1)))
    ram_lak = np.abs(k_freq) * hann_window
    
    # Filter sinogram in Fourier space
    sinogram_fft = np.fft.fft(sinogram, axis=1)
    filtered_fft = sinogram_fft * ram_lak[None, :]
    filtered_sinogram = np.real(np.fft.ifft(filtered_fft, axis=1))
    
    # 3. Backprojection onto 2D phase-space grid (x_out, p_out)
    X, P = np.meshgrid(x_out, p_out)
    W_recon = np.zeros_like(X)
    
    d_theta = np.pi / N_angles
    
    for idx, th in enumerate(angles):
        # Projection coordinate: x_th = x * cos(th) + p * sin(th)
        x_proj = X * np.cos(th) + P * np.sin(th)
        # Interpolate from filtered sinogram
        # Fast nearest/linear interpolation:
        bin_idx = np.clip(np.round((x_proj - x_bins[0]) / dx).astype(int), 0, N_bins - 1)
        W_recon += filtered_sinogram[idx, bin_idx]
        
    # Scaling factor
    W_recon = W_recon * (d_theta / (4.0 * np.pi**2))
    
    # Renormalize to unit integral
    dx_out = x_out[1] - x_out[0]
    dp_out = p_out[1] - p_out[0]
    W_recon /= np.sum(W_recon) * dx_out * dp_out
    
    return W_recon

# Reconstruct 2D Wigner function from the squeezed vacuum sinogram
x_grid_out = np.linspace(-3.5, 3.5, 81)
p_grid_out = np.linspace(-3.5, 3.5, 81)

W_fbp_recon = inverse_radon_fbp(sinogram_sq, angles, x_quad, x_grid_out, p_grid_out)

# Compute true analytical Wigner function for comparison
def true_wigner_sq(r, xvec, pvec):
    X, P = np.meshgrid(xvec, pvec)
    return (1.0 / np.pi) * np.exp(-X**2 * np.exp(2*r) - P**2 * np.exp(-2*r))

W_fbp_true = true_wigner_sq(r_sq, x_grid_out, p_grid_out)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# True Wigner
cf1 = ax1.contourf(x_grid_out, p_grid_out, W_fbp_true, levels=np.linspace(0, 0.35, 45), cmap='plasma')
ax1.set_aspect('equal')
ax1.set_xlim(-3, 3)
ax1.set_ylim(-3, 3)
ax1.set_xlabel(r"Quadrature $x$")
ax1.set_ylabel(r"Quadrature $p$")
ax1.set_title(r"True Wigner Function $W(x, p)$")
plt.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04)

# Reconstructed Wigner via FBP
cf2 = ax2.contourf(x_grid_out, p_grid_out, W_fbp_recon, levels=np.linspace(0, 0.35, 45), cmap='plasma')
ax2.set_aspect('equal')
ax2.set_xlim(-3, 3)
ax2.set_ylim(-3, 3)
ax2.set_xlabel(r"Quadrature $x$")
ax2.set_ylabel(r"Quadrature $p$")
ax2.set_title("FBP Reconstructed Wigner from 60 Sinogram Projections")
plt.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# Compute reconstruction overlap (fidelity metric)
dx_o = x_grid_out[1] - x_grid_out[0]
overlap_fbp = 2.0 * np.pi * np.sum(W_fbp_recon * W_fbp_true) * (dx_o**2)
print(f"FBP Reconstruction Overlap Metric: {overlap_fbp:.4f} (High-fidelity phase space recovery!)")
assert overlap_fbp > 0.85, "FBP reconstruction overlap failed!"


## 3. Maximum Likelihood Quantum State Reconstruction (Hradil MaxLik)

While Filtered Backprojection is fast and elegant, real laboratories collect a finite stream of discrete homodyne detection events $\{ (x_j, \theta_j) \}_{j=1}^M$.

To guarantee that the reconstructed density matrix is **unconditionally positive semidefinite** ($\hat{\rho} \ge 0$), we implement the **Hradil $R\rho R$ iterative algorithm**:

---

### Mathematical Steps of the $R\rho R$ Algorithm:

1. **Quadrature Projector Basis:**
   For a measured outcome $(x_j, \theta_j)$, the measurement projector onto Fock space is:
   $$
   \Pi_j = |x_j, \theta_j\rangle\langle x_j, \theta_j|
   $$
   where $|x_j, \theta_j\rangle = e^{-i\theta_j \hat{n}} |x_j\rangle = \sum_{n=0}^{N-1} e^{-i\theta_j n} \phi_n(x_j) |n\rangle$.
2. **Likelihood Operator $\hat{R}(\hat{\rho})$:**
   At iteration $k$, construct the operator:
   $$
   \hat{R}(\hat{\rho}^{(k)}) = \frac{1}{M} \sum_{j=1}^M \frac{\Pi_j}{\mathrm{Tr}\left[ \hat{\rho}^{(k)} \Pi_j \right]}
   $$
3. **State Update:**
   $$
   \hat{\rho}^{(k+1)} = \frac{\hat{R} \, \hat{\rho}^{(k)} \, \hat{R}}{\mathrm{Tr}\left[ \hat{R} \, \hat{\rho}^{(k)} \, \hat{R} \right]}
   $$
Because $\hat{R}$ is Hermitian, congruence transformation $\hat{R}\hat{\rho}\hat{R}$ **strictly preserves positive semidefiniteness** at every single step!

4. **Stopping Criterion:**
   When the state reaches the maximum likelihood estimate, $\hat{R}(\hat{\rho}) \to \hat{I}$. We stop when the operator distance $\|\hat{R} - \hat{I}\| < \epsilon$ or maximum iterations are reached.


In [ ]:
# Implementation of Iterative Maximum Likelihood (MaxLik) Tomography

def maxlik_tomography(sinogram, angles, x_bins, N_cutoff=15, max_iters=40):
    """Reconstructs density matrix rho using Hradil R*rho*R iterative algorithm from sinogram data."""
    N_angles, N_bins = sinogram.shape
    dx = x_bins[1] - x_bins[0]
    
    # Construct Fock-basis spatial eigenfunctions phi_n(x)
    phi_x = harmonic_quadrature_basis(N_cutoff, x_bins)
    
    # Precompute projectors Pi(th, x) = |x_th><x_th|
    # projector vector: v_{th, x}[n] = exp(-i * th * n) * phi_n(x)
    # Weight each projector by sinogram probability P(th, x) * dx * dtheta
    d_th = np.pi / N_angles
    
    # Initialize rho as maximally mixed state (identity / N)
    rho = np.eye(N_cutoff, dtype=np.complex128) / N_cutoff
    
    # Precompute all projector outer products to speed up iteration
    # To optimize, we sum over angles and bins:
    print(f"Running MaxLik iteration (N_cutoff = {N_cutoff}, iters = {max_iters})...")
    
    for it in range(max_iters):
        R_mat = np.zeros((N_cutoff, N_cutoff), dtype=np.complex128)
        
        for idx_th, th in enumerate(angles):
            rot = np.exp(-1j * th * np.arange(N_cutoff))
            # Matrix of rotated vectors for all x bins: shape (N_cutoff, N_bins)
            v_rot = rot[:, None] * phi_x
            
            # Probability for current rho: p_calc = diag(v_rot.conj().T @ rho @ v_rot)
            p_calc = np.real(np.sum((v_rot.conj().T @ rho) * v_rot.T, axis=1))
            p_calc = np.maximum(1e-12, p_calc)
            
            # Ratio of measured to calculated probability: p_meas / p_calc
            ratio = sinogram[idx_th] / p_calc
            
            # Accumulate into R_mat
            # v_weighted = v_rot * sqrt(ratio * dx * d_th)
            v_weighted = v_rot * np.sqrt(ratio * dx * d_th)
            R_mat += v_weighted @ v_weighted.T.conj()
            
        # Update step: rho_next = R @ rho @ R / Tr(R @ rho @ R)
        R_rho_R = R_mat @ rho @ R_mat
        tr_val = np.real(np.trace(R_rho_R))
        rho = R_rho_R / tr_val
        
    return rho

# Run MaxLik reconstruction on the Squeezed Vacuum state
N_recon_dim = 15
rho_recon = maxlik_tomography(sinogram_sq, angles, x_quad, N_cutoff=N_recon_dim, max_iters=25)

# Benchmark: Quantum State Fidelity F = <psi_true | rho_recon | psi_true>
psi_true_trunc = psi_target[:N_recon_dim]
psi_true_trunc /= np.linalg.norm(psi_true_trunc)

fidelity_maxlik = np.real(np.vdot(psi_true_trunc, rho_recon @ psi_true_trunc))
purity_recon = np.real(np.trace(rho_recon @ rho_recon))

print(f"--- Maximum Likelihood Tomography Results ---")
print(f"Reconstructed State Purity Tr(rho^2): {purity_recon:.6f} (Expected near 1.0 for pure state)")
print(f"Quantum State Fidelity F:            {fidelity_maxlik:.6f} (Target > 0.95)")

assert fidelity_maxlik > 0.95, "MaxLik fidelity benchmark failed!"
print("MaxLik tomography successfully converged to a physical quantum state with >95% fidelity!")


In [ ]:
# Comparative Visualization: True vs. Reconstructed Density Matrix Elements

n_sub_dim = 8

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: True Density Matrix
rho_true_sub = np.outer(psi_true_trunc[:n_sub_dim], psi_true_trunc[:n_sub_dim].conj())
im1 = ax1.imshow(np.real(rho_true_sub), cmap='coolwarm', vmin=-0.3, vmax=0.7)
ax1.set_title(r"True Density Matrix $\mathrm{Re}(\hat{\rho}_{\mathrm{true}})$")
ax1.set_xlabel("Fock Column $m$")
ax1.set_ylabel("Fock Row $n$")
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

# Annotate values
for i in range(n_sub_dim):
    for j in range(n_sub_dim):
        val = np.real(rho_true_sub[i, j])
        if abs(val) > 0.05:
            ax1.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=8)

# Panel 2: MaxLik Reconstructed Density Matrix
rho_recon_sub = np.real(rho_recon[:n_sub_dim, :n_sub_dim])
im2 = ax2.imshow(rho_recon_sub, cmap='coolwarm', vmin=-0.3, vmax=0.7)
ax2.set_title(r"MaxLik Reconstructed $\mathrm{Re}(\hat{\rho}_{\mathrm{recon}})$" + f" (Fidelity = {fidelity_maxlik*100:.1f}%)")
ax2.set_xlabel("Fock Column $m$")
ax2.set_ylabel("Fock Row $n$")
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

for i in range(n_sub_dim):
    for j in range(n_sub_dim):
        val = rho_recon_sub[i, j]
        if abs(val) > 0.05:
            ax2.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Interactive Exploration: Tomographic Reconstruction Explorer

def explore_tomography(n_projections=30, squeeze_r=0.5):
    """Interactive demonstration of how projection count affects FBP reconstruction."""
    N_sim = 20
    a = np.diag(np.sqrt(np.arange(1, N_sim)), k=1)
    S = la.expm(0.5 * squeeze_r * (a@a - a.T.conj()@a.T.conj()))
    vac = np.zeros(N_sim, dtype=np.complex128); vac[0] = 1.0
    psi_sim = S @ vac
    
    th_angles = np.linspace(0, np.pi, n_projections, endpoint=False)
    x_bins = np.linspace(-3.5, 3.5, 70)
    sino = simulate_homodyne_sinogram(psi_sim, th_angles, x_bins)
    
    x_out = np.linspace(-3.0, 3.0, 60)
    W_rec = inverse_radon_fbp(sino, th_angles, x_bins, x_out, x_out)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 1. Sinogram
    ax1.imshow(sino, extent=[x_bins[0], x_bins[-1], 0, np.pi], aspect='auto', cmap='plasma', origin='lower')
    ax1.set_xlabel("Quadrature x")
    ax1.set_ylabel(r"Angle $\theta$ (rad)")
    ax1.set_title(rf"Sinogram ({n_projections} Projections)")
    
    # 2. Reconstructed Wigner
    cf = ax2.contourf(x_out, x_out, W_rec, levels=np.linspace(0, 0.35, 40), cmap='plasma')
    ax2.set_aspect('equal')
    ax2.set_xlabel("x")
    ax2.set_ylabel("p")
    ax2.set_title(rf"Reconstructed Wigner ($r={squeeze_r:.2f}$)")
    plt.colorbar(cf, ax=ax2, fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_tomography,
             n_projections=IntSlider(min=8, max=60, step=4, value=30, description='Projections:'),
             squeeze_r=FloatSlider(min=0.2, max=0.8, step=0.1, value=0.5, description='Squeezing r:'))
else:
    # Static execution fallback for headless environments
    explore_tomography(n_projections=30, squeeze_r=0.5)


## 4. Self-Check Exercises and Pedagogical Challenges

Test your mastery of quantum state tomography and reconstruction algorithms.

---

### Challenge 1: The Fourier Slice Theorem
1. Prove mathematically that the 1D Fourier transform of the homodyne marginal $\mathcal{P}_\theta(x_\theta) = \int W(x_\theta\cos\theta - p_\theta\sin\theta, x_\theta\sin\theta + p_\theta\cos\theta) \, dp_\theta$ gives a radial slice of the 2D characteristic function $\chi(k\cos\theta, k\sin\theta; 0)$.
2. Explain why the Ram-Lak ramp filter $|k|$ is required in Filtered Backprojection.

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
The 1D Fourier transform of the marginal is:
$$
\tilde{\mathcal{P}}_\theta(k) = \int_{-\infty}^\infty \mathcal{P}_\theta(x_\theta) e^{-i k x_\theta} \, dx_\theta
$$
Substitute the definition of $\mathcal{P}_\theta(x_\theta) = \iint W(x, p) \delta(x\cos\theta + p\sin\theta - x_\theta) \, dx \, dp$:
$$
\tilde{\mathcal{P}}_\theta(k) = \iint_{-\infty}^\infty dx \, dp \, W(x, p) \int_{-\infty}^\infty dx_\theta \, \delta(x\cos\theta + p\sin\theta - x_\theta) e^{-i k x_\theta}
$$
The integral over $x_\theta$ evaluates to:
$$
\tilde{\mathcal{P}}_\theta(k) = \iint_{-\infty}^\infty W(x, p) \exp\left[ -i k (x\cos\theta + p\sin\theta) \right] \, dx \, dp
$$
Recalling that the characteristic function $\chi(\xi_x, \xi_p; 0)$ is the 2D Fourier transform of the Wigner function with $\xi_x = k\cos\theta$ and $\xi_p = k\sin\theta$:
$$
\tilde{\mathcal{P}}_\theta(k) = \chi(k\cos\theta, k\sin\theta; 0) \quad \blacksquare
$$
Every 1D homodyne projection at angle $\theta$ provides a radial line through the center of 2D Fourier space!

**Part 2 Solution:**
In polar coordinates, the 2D volume element is $d^2 k = k \, dk \, d\theta$.
The density of sampling points in 2D Fourier space is highest near the origin ($k \to 0$) and thins out as $1/|k|$ at high frequencies. The Ram-Lak filter $|k|$ acts as the polar Jacobian determinant, weighting each Fourier component by $|k|$ to restore flat, uniform 2D frequency coverage and prevent low-frequency blurring!
</details>

---

### Challenge 2: Quantum State Fidelity Benchmark
Using QuTiP or NumPy, verify that the fidelity between a pure state $|\psi\rangle$ and a reconstructed density matrix $\hat{\rho}$ satisfies $\mathcal{F} = \langle\psi|\hat{\rho}|\psi\rangle \le 1.0$, with equality *if and only if* $\hat{\rho} = |\psi\rangle\langle\psi|$.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

```python
# Fidelity properties verification
print(f"Computed MaxLik Fidelity: {fidelity_maxlik:.6f}")
assert 0.0 <= fidelity_maxlik <= 1.000001, "Fidelity out of bounds!"
assert fidelity_maxlik > 0.95, "State reconstruction fidelity should exceed 95%!"
print("Quantum state fidelity validated within physical bounds [0, 1]!")
```
</details>


## 5. Module 2 Grand Capstone Summary and Syllabus Matrix

Congratulations! You have completed **Module 2: Phase Space Distributions & Non-Classicality**.

---

### Master Comparison: The Phase Space Representation Suite

| Distribution | Symbol | Ordering ($s$) | Mathematical Character | Non-Classicality Signature | Experimental Measurement |
| :--- | :---: | :---: | :--- | :--- | :--- |
| **Glauber-Sudarshan** | $P(\alpha)$ | $+1$ (Normal) | Singular (derivatives of $\delta$) | Negative infinities | Kiesel-Vogel Filter $P_\Omega$ |
| **Wigner** | $W(x, p)$ | $0$ (Weyl) | Smooth, real, bounded | Negative regions ($W < 0$) | Homodyne Tomography (FBP/MaxLik) |
| **Husimi** | $Q(\alpha)$ | $-1$ (Anti-normal) | Smooth, strictly non-negative | Zeros of $Q(\alpha)$ | 8-Port Homodyne (Heterodyne) |

---

### Key Takeaways of Module 2:
1. **The Phase-Space Trinity:** The $s$-parameterized hierarchy $P \to W \to Q$ formalizes the trade-off between quantum phase-space resolution and mathematical positivity.
2. **Hudson's Theorem & Wigner Negativity:** Pure Gaussian states are uniquely non-negative; all non-Gaussian pure states possess Wigner negativity, providing an indispensable fuel for quantum computing advantage.
3. **Macroscopic Superpositions:** Cat states exhibit sub-Planck interference fringes in phase space, unlocking Heisenberg-limited phase sensing while suffering accelerated environmental decoherence ($\Gamma_{\text{dec}} = 2\gamma\bar{n}$).
4. **Quantum Tomography:** Homodyne detection combined with Maximum Likelihood estimation (Hradil MaxLik) reconstructs complete, physically valid quantum states from raw experimental quadrature histograms.

---

### Primary References
- **Smithey, D. T., Beck, M., Raymer, M. G., & Faridani, A.** (1993). *Measurement of the Wigner distribution and the density matrix of a light mode using optical homodyne tomography: Application to squeezed states and the vacuum*. Physical Review Letters, 70(9), 1244–1247.
- **Hradil, Z.** (1997). *Quantum-state estimation*. Physical Review A, 55(3), R1561–R1564.
- **Lvovsky, A. I., & Raymer, M. G.** (2009). *Continuous-variable optical quantum-state tomography*. Reviews of Modern Physics, 81(1), 299–332.
- **Leonhardt, U.** (1997). *Measuring the Quantum State of Light*. Cambridge University Press. Chapter 5: Quantum State Tomography.

---

### Forward Look: Module 3 — Light-Matter Interaction & Cavity QED
In **Module 3**, we couple our quantized light modes to two-level atoms and cavity architectures:
- Notebook 3.1: Two-Level Atoms and Optical Bloch Equations
- Notebook 3.2: Rabi Oscillations and Semiclassical Light
- Notebook 3.3: The Jaynes-Cummings Hamiltonian
- Notebook 3.4: Quantum Rabi Model and Dressed States
- Notebook 3.5: Collapse and Revival of Quantum Inversion
- Notebook 3.6: The Mollow Triplet and Resonance Fluorescence
- Notebook 3.7: Electromagnetically Induced Transparency (EIT) and Dark States
